# Week 4 lab

Part 1: Data Preparation

In [ ]:
import warnings

import contextily as cx
import dask.array as da
import matplotlib.pyplot as plt
import numpy as np
import odc.stac
import pandas as pd
import planetary_computer
import pyproj
import pystac_client
import seaborn as sns
import xarray as xr
from dask.distributed import Client
from dask_ml.cluster import KMeans
from dask_ml.linear_model import LogisticRegression
from IPython.display import Image
from matplotlib.patches import Rectangle
from skimage.filters import threshold_otsu
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

In [ ]:
#connecting to planetary computer STAC API
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

#boundary box parameters
bbox = [-118.89, 38.54, -118.57, 38.84]  # Region over a lake in Nevada, USA
datetime = "2017-06-01/2017-09-30"  # Summer months of 2017
collection = "landsat-c2-l2"
platform = "landsat-8"
cloudy_less_than = 1  # percent

#only taking the ones that have low cloud cover for more accurate
search = catalog.search(
    collections=["landsat-c2-l2"],
    bbox=bbox,
    datetime=datetime,
    query={"eo:cloud_cover": {"lt": cloudy_less_than}, "platform": {"in": [platform]}},
)
items = search.get_all_items()
print(f"Returned {len(items)} Items:")
[[i, item.id] for i, item in enumerate(items)]

In [ ]:
item = items[1]

In [ ]:
Image(url=item.assets["rendered_preview"].href)

In [ ]:
assets = []
for _, asset in item.assets.items():
    try:
        assets.append(asset.extra_fields["eo:bands"][0])
    except:
        pass

cols_ordered = [
    "common_name",
    "description",
    "name",
    "center_wavelength",
    "full_width_half_max",
]
bands = pd.DataFrame.from_dict(assets)[cols_ordered]
bands

In [ ]:
ds_2017 = odc.stac.stac_load(
    [item],
    bands=bands.common_name.values,
    bbox=bbox,
    chunks={},  # <-- use Dask
).isel(time=0)

retain crs Attribute

In [ ]:
prop = item.properties
print(prop)

In [ ]:
epsg = item.properties.get("proj:code")
ds_2017.attrs["crs"] = f"{epsg}"

In [ ]:
ds_2017

In [ ]:
da_2017 = ds_2017.to_array(dim="band")
da_2017

flatten array to two demensional

In [ ]:
flattened_xda = da_2017.stack(z=("x", "y"))  # flatten each band
flattened_t_xda = flattened_xda.transpose("z", "band")
flattened_t_xda

Standardization

In [ ]:
with xr.set_options(keep_attrs=True):
    rescaled_xda = (flattened_t_xda - flattened_t_xda.mean()) / flattened_t_xda.std()
rescaled_xda

## Part 2: K means clustering

In [ ]:
client = Client(processes=False)
client

In [ ]:
X_2017 = client.persist(rescaled_xda)
X_2017.shape

In [ ]:
#kmeans clustering (k=4)
kmeans = KMeans(n_clusters=4, random_state=0, max_iter=100)
kmeans.fit(X_2017)
labels = kmeans.labels_.compute()

template_2017 = flattened_t_xda[:,0]
output_array = template_2017.copy(data=labels)
output_array

In [ ]:
unstacked_2017 = output_array.unstack()
unstacked_2017

In [ ]:
fig, (ax1) = plt.subplots(1, figsize=(12, 5))

# Plot raw
da_2017.sel(band="blue").plot(ax=ax1, cmap="gray")
ax1.set_title("Raw Image (Blue Band)")

# Plot clustering - transpose if needed
#unstacked_2017.T.plot(ax=ax2, cmap="Set3", vmin=0, vmax=3)  # Add .T to transpose
#ax1.set_title("Clustering")

plt.tight_layout()
plt.show()



Supervised Classification with linear model